# RunnableParallel: 데이터 병렬 조작

## 입력 및 출력 조작

`RunnableParallel` 은 시퀀스 내에서 하나의 `Runnable` 의 출력을 다음 `Runnable` 의 입력 형식에 맞게 조작하는 데 유용하게 사용될 수 있습니다.

여기서 prompt에 대한 입력은 "context"와 "question"이라는 키를 가진 map 형태로 예상됩니다.

사용자 입력은 단순히 질문 내용입니다. 따라서 retriever를 사용하여 컨텍스트를 가져오고, 사용자 입력을 "question" 키 아래에 전달해야 합니다.



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["Hailey is an Product Manager who loves programming!"], embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-4o-mini")

retriever_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

retriever_chain.invoke("What is Hailey's occupation?")

"Hailey's occupation is a Product Manager."

다른 `Runnable` 과 함께 `RunnableParallel` 을 구성할 때, 유형 변환이 자동으로 처리되므로 `RunnableParallel` 클래스에서 입력으로 주입되는 dict 입력을 별도 래핑할 필요도 없다는 점에 유의하세요.

아래의 3가지 방식은 모두 동일하게 처리합니다.

```python
# 자체 RunnableParallel 로 래핑됨
1. {"context": retriever, "question": RunnablePassthrough()}

2. RunnableParallel({"context": retriever, "question": RunnablePassthrough()})

3. RunnableParallel(context=retriever, question=RunnablePassthrough())
```


## itemgetter를 단축어로 사용하기

RunnableParallel과 결합할 때 Python의 `itemgetter`를 단축어로 사용하여 map에서 데이터를 추출할 수 있습니다.

- [참고] itemgetter에 대한 자세한 정보는 [Python Documentation](https://docs.python.org/3/library/operator.html#operator.itemgetter)에서 확인할 수 있습니다.

아래 예제에서는 `itemgetter` 를 사용하여 map에서 특정 키를 추출합니다.


In [4]:
from operator import itemgetter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["Hailey is an Product Manager who loves programming!"], embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context" : itemgetter("question") | retriever,
        "question" : itemgetter("question"),
        "language" : itemgetter("language"),
    }
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

chain.invoke({"question": "What is Hailey's occupation?", "language": "Korean"})

'헤일리는 제품 관리자입니다.'

## 병렬처리를 단계별로 이해

`RunnableParallel` 을 사용하면 여러 `Runnable` 을 병렬로 실행하고, 이러한 `Runnable` 의 출력을 맵(map)으로 반환하는 것이 쉬워집니다.


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

model =ChatOpenAI()

capital_chain = (
    ChatPromptTemplate.from_template("{country}의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain = (
    ChatPromptTemplate.from_template("{country}의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

map_chain = RunnableParallel(capital=capital_chain, area= area_chain)

map_chain.invoke({"country": "대한민국"})

{'capital': '대한민국의 수도는 서울입니다.',
 'area': '대한민국의 총 면적은 약 100,363 제곱 킬로미터이며, 대한민국은 세계에서 108번째로 면적이 작은 국가 중 하나입니다.'}

아래와 같이 chain 별로 입력 템플릿의 변수가 달라도 상관없이 실행 가능합니다.


In [5]:
capital_chain2 = (
    ChatPromptTemplate.from_template("{country1}의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

area_chain2 = (
    ChatPromptTemplate.from_template("{country2}의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

map_chain2 = RunnableParallel(capital=capital_chain2, area=area_chain2)

map_chain2.invoke({"country1": "대한민국", "country2": "미국"})

NameError: name 'RunnableParallel' is not defined

## 병렬 처리

`RunnableParallel` 은 맵에 있는 각 `Runnable` 이 병렬로 실행되기 때문에 독립적인 프로세스를 병렬로 실행하는 데에도 유용합니다.

예를 들어, 앞서 살펴본 `area_chain`, `capital_chain`, `map_chain`은 `map_chain`이 다른 두 체인을 모두 실행함에도 불구하고 **거의 동일한 실행 시간** 을 가지는 것을 확인할 수 있습니다.


In [11]:
%%timeit

# 면저을 묻는 체인을 호출하고 실행 시간을 측정합니다.
area_chain.invoke({"country": "대한민국"})

1.22 s ± 499 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit

# 수도를 묻는 체인을 호출하고 실행 시간을 측정합니다.
capital_chain.invoke({"country": "대한민국"})

1.45 s ± 382 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit

# Parallel 하게 구성된 체인을 호출하고 실행 시간을 측정합니다.
map_chain.invoke({"country": "대한민국"})

1.7 s ± 373 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
